# 02 — TensorRT Inference Sweep

For each trained checkpoint (`seed_1`, `seed_2`, `seed_42`), sweep all 5 precisions × 4 input bit-depths.

ONNX exports (base + QDQ) are assumed to already exist from `00_qdq_export.ipynb`.

Results saved under `runs/val_infer/<seed>/`, averaged at the end.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
if str(PYFILES) not in sys.path:
    sys.path.insert(0, str(PYFILES))

In [ ]:
import torch
import pandas as pd

from src.config import ExperimentConfig, with_overrides
from src.runner import run_experiment
from utils.utils import load_runs, flatten_runs

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.3f}".format)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
CKPT_ROOT = PROJECT_ROOT / "checkpoints"
ONNX_ROOT = PROJECT_ROOT / "onnx"
INPUT_BITS = [8, 4, 2, 1]
TRT_PRECISIONS = ["fp32", "fp16", "int8", "fp8", "int4"]

# Discover (bits, seed) pairs from checkpoints
experiments = {}
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if seed_dir.is_dir() and (seed_dir / "best.pth").exists():
            seed = seed_dir.name
            seed_num = seed.split("_")[-1]
            experiments[(bits, seed)] = {
                "ckpt": str(seed_dir / "best.pth"),
                "seed_num": seed_num,
            }

def make_base(bits, seed, seed_num):
    return ExperimentConfig(
        backend="tensorrt",
        device="cuda",
        batch_size=1,
        seed=42,
        num_eval_batches=500,
        trt_static_shape=True,
        trt_workspace_mb=2048,
        onnx_root=str(ONNX_ROOT / f"fp32_{bits}bit"),
        engine_root=str(PROJECT_ROOT / "engines" / f"fp32_{bits}bit" / seed),
        trt_onnx_prefix=f"resnet_{seed_num}",
        output_root=f"../runs/val_infer/fp32_{bits}bit/{seed}",
    )

print("Experiments found:")
for (bits, seed), info in experiments.items():
    print(f"  {bits}bit / {seed}: {info['ckpt']}")


## FP32

In [ ]:
fp32_records = []
print("TRT Precision: fp32")
for (bits, seed), info in experiments.items():
    onnx_base = ONNX_ROOT / f"fp32_{bits}bit" / f"resnet_{info['seed_num']}.onnx"
    if not onnx_base.exists():
        print(f"  SKIP {bits}bit / {seed}: ONNX not exported yet")
        continue

    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="fp32", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=info["ckpt"])
    fp32_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

## FP16

In [ ]:
fp16_records = []
print("TRT Precision: fp16")
for (bits, seed), info in experiments.items():
    onnx_base = ONNX_ROOT / f"fp32_{bits}bit" / f"resnet_{info['seed_num']}.onnx"
    if not onnx_base.exists():
        print(f"  SKIP {bits}bit / {seed}: ONNX not exported yet")
        continue

    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="fp16", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=info["ckpt"])
    fp16_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

## INT8

In [ ]:
int8_records = []
print("TRT Precision: int8")
for (bits, seed), info in experiments.items():
    onnx_base = ONNX_ROOT / f"fp32_{bits}bit" / f"resnet_{info['seed_num']}.onnx"
    if not onnx_base.exists():
        print(f"  SKIP {bits}bit / {seed}: ONNX not exported yet")
        continue

    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="int8", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=info["ckpt"])
    int8_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

## FP8

In [ ]:
fp8_records = []
print("TRT Precision: fp8")
for (bits, seed), info in experiments.items():
    onnx_base = ONNX_ROOT / f"fp32_{bits}bit" / f"resnet_{info['seed_num']}.onnx"
    if not onnx_base.exists():
        print(f"  SKIP {bits}bit / {seed}: ONNX not exported yet")
        continue

    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="fp8", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=info["ckpt"])
    fp8_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

## INT4

In [ ]:
int4_records = []
print("TRT Precision: int4")
for (bits, seed), info in experiments.items():
    onnx_base = ONNX_ROOT / f"fp32_{bits}bit" / f"resnet_{info['seed_num']}.onnx"
    if not onnx_base.exists():
        print(f"  SKIP {bits}bit / {seed}: ONNX not exported yet")
        continue

    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="int4", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=info["ckpt"])
    int4_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

## All Results (per seed)

In [ ]:
all_rows = []
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if not seed_dir.is_dir():
            continue
        run_dir = f"../runs/val_infer/fp32_{bits}bit/{seed_dir.name}"
        runs = load_runs(run_dir, status="ok")
        for r in flatten_runs(runs):
            r["seed"] = seed_dir.name
            r["input_bits_trained"] = bits
            all_rows.append(r)

df = pd.DataFrame(all_rows)
df_trt = df[df["cfg.backend"] == "tensorrt"]

display_cols = [
    "seed", "input_bits_trained", "cfg.model_precision",
    "res.top1_acc", "res.top5_acc", "res.infer_ms_avg", "res.throughput_infer_sps",
]
df_trt[display_cols].sort_values(
    ["input_bits_trained", "seed", "cfg.model_precision"],
    ascending=[False, True, True],
).reset_index(drop=True)


## Averaged Results (across seeds)

In [ ]:
avg_df = df_trt.groupby(["cfg.backend", "cfg.model_precision", "input_bits_trained"]).agg(
    top1_mean=("res.top1_acc", "mean"),
    top1_std=("res.top1_acc", "std"),
    top5_mean=("res.top5_acc", "mean"),
    top5_std=("res.top5_acc", "std"),
    infer_ms_mean=("res.infer_ms_avg", "mean"),
    infer_ms_std=("res.infer_ms_avg", "std"),
    throughput_mean=("res.throughput_infer_sps", "mean"),
).round(3).reset_index()

avg_df = avg_df.sort_values(
    ["cfg.model_precision", "input_bits_trained"],
    ascending=[True, False],
).reset_index(drop=True)
avg_df


In [ ]:
avg_df[["cfg.model_precision", "input_bits_trained",
        "top1_mean", "top1_std", "top5_mean", "top5_std",
        "infer_ms_mean", "infer_ms_std"]].assign(
    top1=lambda d: d.apply(lambda r: f"{r['top1_mean']:.2f} ± {r['top1_std']:.2f}", axis=1),
    top5=lambda d: d.apply(lambda r: f"{r['top5_mean']:.2f} ± {r['top5_std']:.2f}", axis=1),
    infer_ms=lambda d: d.apply(lambda r: f"{r['infer_ms_mean']:.3f} ± {r['infer_ms_std']:.3f}", axis=1),
)[["cfg.model_precision", "input_bits_trained", "top1", "top5", "infer_ms"]]


In [ ]:
out_path = PROJECT_ROOT / "results" / "tensorrt_avg_results.json"
avg_df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")